In [ ]:
# 1) Mount Drive + persistent paths (survive disconnects)
from google.colab import drive; drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/attackdro'
for sub in ('results','checkpoints','data'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print('Drive workspace ready:', DRIVE)

In [ ]:
# 2) Clone the repo and pin the exact reviewed snapshot (dab21df)
import os, subprocess
SHA  = 'dab21df'                      # card-pb-fixes HEAD: RAMP recipe + 10/10/10 train + 20/20/100 eval + nested layout
REPO = 'github.com/anhkiet287/attackdro.git'
try:
    from google.colab import userdata; PAT = userdata.get('GH_PAT')
except Exception:
    PAT = None
if not PAT:
    from getpass import getpass; PAT = getpass('GitHub PAT (repo read): ')
url = f'https://{PAT}@{REPO}'


In [ ]:
subprocess.run(['git','fetch','-q','origin','card-pb-fixes'], check=True)
subprocess.run(['git','checkout','-q',SHA], check=True)
head = subprocess.run(['git','rev-parse','--short','HEAD'], capture_output=True, text=True).stdout.strip()
assert head.startswith(SHA[:7]), f'HEAD {head} != {SHA}'
print('repo @', head, '(pinned)')

In [ ]:
import shutil

if not os.path.isdir('/content/attackdro/.git'):
    # Clean up the directory if it exists but is not a valid git repo
    if os.path.exists('/content/attackdro'):
        shutil.rmtree('/content/attackdro')
    try:
        subprocess.run(['git','clone','-q',url,'/content/attackdro'], check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print("Git Clone Failed!")
        print("Error message:", e.stderr)
        raise
os.chdir('/content/attackdro')

In [ ]:
# 3) Deps (torch/torchvision already on Colab; match requirements.txt for the rest)
!pip -q install numpy pyyaml tqdm matplotlib wandb==0.28.0
!pip -q install "autoattack @ git+https://github.com/fra31/auto-attack.git"
print('deps installed')

In [ ]:
# 4) GUARD — refuse to run on the wrong protocol / recipe (catches a bad pin)
import sys; sys.path.insert(0, 'src')
from robustdro.utils.io import load_config
b = load_config('configs/base.yaml'); tm = b['threat_model']; ea = b['eval_attack']
assert abs(tm['linf']['eps'] - 8/255) < 1e-9, f"WRONG eps {tm['linf']['eps']} (expected 8/255) — wrong branch/SHA, STOP"
assert tm['l2']['eps'] == 0.5 and tm['l1']['eps'] == 12.0, f"l2/l1 mismatch: {tm}"
assert (ea['linf']['steps'], ea['l2']['steps'], ea['l1']['steps']) == (20, 20, 100), f"eval budget != 20/20/100 (screening): {ea}"
atk = [a['steps'] for a in load_config('configs/attackdro.yaml')['train']['attacks']]
assert atk == [10, 10, 10], f"train inner-max != 10/10/10 (RAMP-matched): {atk}"
print("GUARD OK: eps=(8/255, 0.5, 12) · train attack 10/10/10 · eval 20/20/100 · pin carries the RAMP-recipe code")


In [ ]:
# 5) W&B online (project attackdro-union, entity=default; same schema as local)
import os
os.environ['WANDB_MODE'] = 'online'
try:
    from google.colab import userdata; K = userdata.get('WANDB_API_KEY')
except Exception:
    K = None
if not K:
    from getpass import getpass; K = getpass('W&B API key: ')
os.environ['WANDB_API_KEY'] = K
import wandb; wandb.login(key=K)
print('W&B online -> attackdro-union (entity = API-key default)')

In [ ]:
import hashlib, os
DATA = f'{DRIVE}/data'
h = hashlib.md5(open(f'{DATA}/cifar-10-python.tar.gz','rb').read()).hexdigest()
print(h, '->', 'OK, will NOT re-download' if h=='c58f30108f718f92721af3b95e74349a' else 'MISMATCH — torchvision will re-download')


In [ ]:
# 6) SMOKE both configs first (2 steps, wandb disabled) — proves data->model->backward->eval path.
#    Also triggers the one-time CIFAR-10 download into the Drive cache.
import subprocess, os
DATA = f'{DRIVE}/data'
env = dict(os.environ, PYTHONPATH='src', WANDB_MODE='disabled')
for cfg in ('configs/avg_frozen.yaml','configs/bindaware.yaml'):
    print('=== SMOKE', cfg, '===', flush=True)
    subprocess.run(['python','scripts/train.py','--config',cfg,'--smoke',
                    '--wandb-mode','disabled','--set',f'dataset.root={DATA}'],
                   check=True, env=env)
print('SMOKE OK — both anchor configs run end-to-end; CIFAR cached on Drive')

In [ ]:
# 7) Train + eval the Phase-1 anchor rows @ RAMP recipe, STOP@ep50, 1 seed.
#    Writes DIRECTLY to Drive in the NESTED per-run layout (matches the 5070ti runs), so
#    Kiet's dashboard reads them identically. Disconnect-proof: skip-if-done + resume.
import subprocess, os, glob
DATA = f'{DRIVE}/data'
RES  = f'{DRIVE}/results'          # RES/<run>/s<seed>/{train,eval}.json + ckpt/ + run_meta.json
os.makedirs(RES, exist_ok=True)
env = dict(os.environ, PYTHONPATH='src', WANDB_MODE='online')
# RAMP recipe (Phase-1 develop = stop@ep50, pre-drop): lr 0.05 flat (drop is at ep70), 80ep
# schedule but we run 50; save/10; train attack 10/10/10 (inherited); eval 20/20/100 (base).
RECIPE = ['--epochs', '50', '--set', 'train.lr=0.05', '--set', 'train.milestones=[70]',
          '--set', f'results_dir={RES}/', '--set', f'dataset.root={DATA}',
          '--set', "wandb.tags=['device:colab']"]
JOBS = [('configs/avg_frozen.yaml', 'avg_frozen_8255', 'F4 uniform anchor'),
        ('configs/bindaware.yaml',  'bindaware_3a_8255', '3a signal-axis (weight_signal=robust_acc)')]
SEED = 0
for cfg, run, note in JOBS:
    sd = f'{RES}/{run}/s{SEED}'
    eval_out, last = f'{sd}/eval.json', f'{sd}/ckpt/last.pt'
    if os.path.exists(eval_out):
        print('SKIP (done):', run, flush=True); continue
    if not os.path.exists(last):
        ep_ck = sorted(glob.glob(f'{sd}/ckpt/ep*.pt'))          # resume mid-train after a Colab drop
        resume = ['--resume', ep_ck[-1]] if ep_ck else []
        tag = f' (resume {os.path.basename(ep_ck[-1])})' if ep_ck else ''
        print(f'=== TRAIN {run} ({note}) -> ep50 @ RAMP recipe{tag} ===', flush=True)
        subprocess.run(['python', 'scripts/train.py', '--config', cfg, '--seed', str(SEED),
                        '--run-name', run, '--wandb-mode', 'online'] + RECIPE + resume, check=True, env=env)
    else:
        print('Training complete on Drive — eval only:', run, flush=True)
    print(f'=== EVAL {run} (screening APGD 20/20/100, n=1000) -> {eval_out} ===', flush=True)
    subprocess.run(['python', 'scripts/evaluate.py', '--config', cfg, '--checkpoint', last,
                    '-n', '1000', '--version', 'apgd', '--run-name', run, '--seed', str(SEED),
                    '--tier', 'in-house', '--out', eval_out, '--set', f'dataset.root={DATA}'],
                   check=True, env=env)
    print('DONE', run, '-> Drive + W&B (device:colab, tier:in-house)', flush=True)
print('ALL PHASE-1 ANCHOR RUNS COMPLETE (ep50, 1 seed)')


In [ ]:
# 8) Phase-1 anchor table (reads the nested Drive eval.json)
import json, os
RES = f'{DRIVE}/results'
hdr = f"{'run':20} {'union':>6} {'clean':>6} {'linf':>6} {'l2':>6} {'l1':>6}"
print(hdr + "   (@ep50, RAMP recipe, screening APGD 20/20/100)")
for run in ('avg_frozen_8255', 'bindaware_3a_8255'):
    f = f'{RES}/{run}/s0/eval.json'
    if not os.path.exists(f):
        print(f'{run:20}  — pending'); continue
    m = json.load(open(f))['metrics']; pn = m['per_norm_robust_acc']
    print(f"{run:20} {100*m['worst_union_acc']:6.1f} {100*m['clean_acc']:6.1f} "
          f"{100*pn['linf']:6.1f} {100*pn['l2']:6.1f} {100*pn['l1']:6.1f}")
print('\nCompare vs the 5070ti reactive / predictive @ep50 (same recipe, same budget) and RAMP ep50 = 42.9.')
